# Optimal Number of Clusters

We will now determine the Optimal Number of Clusters (ONC) which will then feed the Nested Clustered Optimisation (NCO) allocator
- First we denoise and detone the correlation
- Iterate over a series of possible number of clusters
- Score each iteration with the silhouette method
- Keep the k number of clusters with the best score

In [16]:
# Imports

import pandas as pd
import numpy as np
from vportfolio import load_data, engine, denoising, clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

## Denoise and detone

In [17]:
# data

data = load_data.load_universe()

# let's take the last date of the research period

decision_date = data["rebal_dates"].iloc[-1]

# calculate the lookback window

window = engine.lookback_window(data["returns"], data["eligible"], decision_date, n=LOOKBACK)

# calculate the correlation matrix

corr = window.corr()

# remove the noise from the correlation matrix 
corr_dn = denoising.denoise_corr(corr, window.shape[0])

To detone the matrix we essentially need to remove the market factor or risk appetite from the matrix. This is done by removing the largest eigenvalue from the matrix

In [18]:
# detone a correlation matrix
# will return a dataframe with the detoned correlation matrix
def detone_corr(corr):
    eigenvals, eigenvec = np.linalg.eigh(corr) # smallest first
    lambda_1 = eigenvals[-1] # market factor: the largest eigenvalue
    v1 = eigenvec[:, -1] # and its eigenvector (last column)
    market_part = lambda_1 * np.outer(v1, v1) # the market's contribution to the matrix
    detoned = corr - market_part # remove it
    d = np.sqrt(np.diag(detoned)) # rescale so the diagonal is 1 again
    detoned = detoned / np.outer(d, d)
    return pd.DataFrame(detoned, index=corr.index, columns=corr.columns)

In [19]:
# testing the function

# 1. always true: symmetric, diagonal 1, no NaNs
rng = np.random.default_rng(0)
random_corr = pd.DataFrame(rng.normal(size=(500, 6))).corr()
result = detone_corr(random_corr)
assert np.allclose(result, result.T)
assert np.allclose(np.diag(result), 1)
assert result.notna().all().all()
# 2. known answer: pure market (all 0.6) → off-diagonals become −1/(n−1) = −0.25 with 5 assets
m = np.full((5, 5), 0.6)
np.fill_diagonal(m, 1)
result = detone_corr(pd.DataFrame(m))
off_diag = ~np.eye(5, dtype=bool)
assert np.allclose(result.values[off_diag], -0.25)
# 3. inputs matter: block structure survives (0.8 within blocks, 0.3 between)
m = np.full((6, 6), 0.3)
m[:3, :3] = 0.8
m[3:, 3:] = 0.8
np.fill_diagonal(m, 1)
result = detone_corr(pd.DataFrame(m))
within = result.values[0, 1] # inside block 1
between = result.values[0, 3] # block 1 vs block 2
assert within > between

print("detone_corr passed all tests")

detone_corr passed all tests


In [20]:
# let's now detone the correlation

corr_dt = detone_corr(corr_dn) 

## ONC

We now need to build the ONC iterator

In [21]:
def onc(corr, max_k=10, n_init=10):
    dist = corr_to_distance(corr)
    max_k = min(max_k, len(corr) - 1)          # silhouettes need fewer clusters than ETFs
    best_score, best_labels = -np.inf, None
    for k in range(2, max_k + 1):
        for seed in range(n_init):
            labels = KMeans(n_clusters=k, n_init=1, random_state=seed).fit(dist).labels_
            sil = silhouette_samples(dist, labels)
            score = ...                         # mean ÷ std
            if score > best_score:
                ...                             # keep score and labels
    return pd.Series(best_labels, index=corr.index)